# Lab 9 — Transfer learning: standing on the shoulders of ImageNet

**Deep Learning for Computer Vision — Day 3** · ECAM, 5MAU · 2026–2027

In industry you rarely have 1 million labelled images. **Transfer learning** reuses a network pre-trained on a large dataset (ImageNet: 1.2 M images, 1 000 classes) and adapts it to your task with a few dozen / hundred images.

**Goals.**
1. Load a pre-trained ResNet-18 and its preprocessing.
2. Compare three strategies: **from scratch**, **linear probe** (frozen backbone, new head), **fine-tuning** (everything trainable, small learning rate).
3. On two tasks: ants vs bees (natural images, close to ImageNet) and our flange inspection with **only 10 images per class** (industrial, far from ImageNet).
4. Use the pre-trained features to find similar images (nearest neighbours).

**Duration:** ~60 min. Each training takes 1–3 min on a laptop CPU (or a few seconds on a Colab GPU: *Runtime → Change runtime type → GPU*).

> *References:* PyTorch tutorial *Transfer learning for computer vision* (ants & bees dataset), https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html · Stanford CS231n, Lecture 6 (*transfer learning*), 2025 · KenN7, *Deep learning for computer vision*, ECAM, course 3 · K. He *et al.*, "Deep residual learning for image recognition", CVPR 2016 · J. Yosinski *et al.*, "How transferable are features in deep neural networks?", NeurIPS 2014. Flange images: synthetic dataset of Lab 7.

In [ ]:
import sys

if "google.colab" in sys.modules:  # Colab: the packages are installed by the next cell
    print("Colab: setup check skipped.")
else:
    # Setup check: are all the packages of requirements.txt installed in this kernel?
    import importlib
    import importlib.util

    importlib.invalidate_caches()
    REQUIRED = {"torch": "torch", "torchvision": "torchvision", "ultralytics": "ultralytics", "numpy": "numpy",
                "matplotlib": "matplotlib", "cv2": "opencv-contrib-python"}
    missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
    assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                         "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
    print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, models, transforms as T
from torchvision.datasets.utils import download_and_extract_archive

DATA = Path("data") if Path("data").exists() else Path("../data")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

if not (DATA / "hymenoptera_data").exists():  # 45 MB, only the first time
    download_and_extract_archive("https://download.pytorch.org/tutorial/hymenoptera_data.zip", DATA)

## 1. A pre-trained network

`torchvision.models` provides architectures with weights trained on ImageNet. **The input must be preprocessed exactly as during pre-training** (size, normalisation with the ImageNet mean and std) — Lab 8, bug 5!

In [ ]:
weights = models.ResNet18_Weights.IMAGENET1K_V1
backbone = models.resnet18(weights=weights)
print(weights.transforms())
print("parameters:", sum(p.numel() for p in backbone.parameters()))
print(backbone.fc)  # the last layer: 512 features -> 1000 ImageNet classes

### Exercise 1.1 — What does ImageNet-ResNet see?
Classify one ants/bees image with the **unmodified** network and print the top-5 ImageNet classes (`weights.meta["categories"]`). Use `weights.transforms()` as preprocessing.

In [ ]:
from PIL import Image

img_path = sorted((DATA / "hymenoptera_data" / "val" / "bees").glob("*.jpg"))[3]
img = Image.open(img_path).convert("RGB")
# TODO
top5 = [(class_name, probability), ...]
plt.imshow(img); plt.axis("off")
for name, prob in top5:
    print(f"{prob:6.3f}  {name}")

## 2. Data and preprocessing

The ants/bees dataset has only **~120 training images per class**.

### Exercise 2.1
Build `train_tfm` (with augmentation: `RandomResizedCrop(224)`, `RandomHorizontalFlip`) and `eval_tfm` (`Resize(256)`, `CenterCrop(224)`), both ending with `ToTensor()` and the **ImageNet normalisation** (`IMAGENET_MEAN`, `IMAGENET_STD`).

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = weights.transforms().mean, weights.transforms().std
# TODO
train_tfm = ... ; eval_tfm = ...
hym_train = datasets.ImageFolder(DATA / "hymenoptera_data" / "train", train_tfm)
hym_val = datasets.ImageFolder(DATA / "hymenoptera_data" / "val", eval_tfm)
print(hym_train.classes, len(hym_train), "train /", len(hym_val), "val")
fig, ax = plt.subplots(1, 6, figsize=(15, 2.8))
for a, i in zip(ax, np.random.default_rng(0).choice(len(hym_val), 6, replace=False)):
    x, y = hym_val[i]
    a.imshow((x.permute(1, 2, 0) * torch.tensor(IMAGENET_STD) + torch.tensor(IMAGENET_MEAN)).clamp(0, 1))
    a.set_title(hym_val.classes[y]); a.axis("off")

## 3. Three strategies

| strategy | weights at start | trained parameters | learning rate |
|---|---|---|---|
| from scratch | random | all | normal (1e-3) |
| linear probe | ImageNet | **only the new head** | normal (1e-3) |
| fine-tuning | ImageNet | all | **small** for the backbone (1e-4), normal for the head |

### Exercise 3.1
Complete `make_model(strategy, n_classes)`. It returns the model and the list of parameter groups for the optimiser.
* Replace `model.fc` by a new `nn.Linear(512, n_classes)`.
* `probe`: set `requires_grad = False` for every parameter **before** creating the new head.
* `finetune`: two groups, `{"params": backbone params, "lr": 1e-4}` and `{"params": head params, "lr": 1e-3}`.

In [ ]:
def make_model(strategy, n_classes):
    # TODO
    ...


m, g = make_model("probe", 2)
n_trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
print("probe: trainable parameters =", n_trainable)
assert n_trainable == 512 * 2 + 2

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x.to(device)).argmax(1).cpu() == y).sum().item()
            n += len(y)
    return correct / n


def train(strategy, train_set, val_set, epochs, seed=0):
    torch.manual_seed(seed)
    model, groups = make_model(strategy, len(train_set.classes))
    model.to(device)
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    loader = torch.utils.data.DataLoader(train_set, batch_size=16, shuffle=True)
    t0 = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        if strategy == "probe":
            model.eval()  # frozen backbone: keep its batch-norm statistics from ImageNet
        for x, y in loader:
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    acc = evaluate(model, torch.utils.data.DataLoader(val_set, batch_size=64))
    print(f"{strategy:9s}: accuracy {acc:.3f}  ({time.perf_counter() - t0:.0f}s)")
    return model, acc


results = {}
for strategy, epochs in [("scratch", 5), ("probe", 5), ("finetune", 5)]:
    results[("ants/bees", strategy)] = train(strategy, hym_train, hym_val, epochs)[1]

## 4. An industrial task with 10 images per class

The flanges of Lab 7, but now we only have **10 labelled images per class** (40 in total, from the production line). The test set has 80 images per class. The images are grey: we repeat the channel 3 times to feed an RGB network.

In [ ]:
parts_train_tfm = T.Compose([T.Grayscale(3), T.Resize(224), T.RandomRotation(180), T.RandomHorizontalFlip(),
                             T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
parts_eval_tfm = T.Compose([T.Grayscale(3), T.Resize(224), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
parts_train = datasets.ImageFolder(DATA / "parts_fewshot" / "train", parts_train_tfm)
parts_test = datasets.ImageFolder(DATA / "parts_fewshot" / "test", parts_eval_tfm)
print(parts_train.classes, len(parts_train), "train /", len(parts_test), "test")

for strategy in ["scratch", "probe", "finetune"]:
    model, acc = train(strategy, parts_train, parts_test, epochs=15)
    results[("flanges, 10/class", strategy)] = acc
    if strategy == "finetune":
        finetuned_parts = model

In [ ]:
tasks = ["ants/bees", "flanges, 10/class"]
strategies = ["scratch", "probe", "finetune"]
x = np.arange(len(tasks))
plt.figure(figsize=(8, 3.5))
for k, s in enumerate(strategies):
    v = [results[(t, s)] for t in tasks]
    plt.bar(x + (k - 1) * 0.27, v, 0.27, label=s)
    for xi, vi in zip(x, v):
        plt.text(xi + (k - 1) * 0.27, vi + 0.01, f"{vi:.2f}", ha="center", fontsize=9)
plt.xticks(x, tasks); plt.ylim(0, 1.1); plt.ylabel("accuracy"); plt.legend(loc="lower right");

**Questions.**
1. Why is training from scratch so bad on both tasks?
2. On ants/bees the linear probe is excellent; on the flanges fine-tuning is clearly better. Explain with the notion of **distance between the source domain (ImageNet) and the target domain**.
3. Lab 7 reached ~97 % with 800 training images and no pre-training. What is the trade-off?

## 5. Pre-trained features as a similarity measure

Remove the classification head: the 512 features of the penultimate layer are an **embedding**. Similar images have close embeddings — useful to search a database, find duplicates, or spot anomalies.

### Exercise 5.1
Compute the embeddings of the flange test images with the fine-tuned network (`model.fc = nn.Identity()`), then show the 5 nearest neighbours (cosine similarity) of a query image.

In [ ]:
embedder = finetuned_parts
embedder.fc = nn.Identity()
embedder.eval()
loader = torch.utils.data.DataLoader(parts_test, batch_size=64)
with torch.no_grad():
    # TODO
    feats = ... (N x 512, L2-normalised) ; labels = ...
q = 5
sims = feats @ feats[q]
nn_idx = sims.argsort(descending=True)[:6]
fig, ax = plt.subplots(1, 6, figsize=(13, 2.6))
for a, i in zip(ax, nn_idx):
    a.imshow(parts_test[i][0][0] * IMAGENET_STD[0] + IMAGENET_MEAN[0], cmap="gray")
    a.set_title(("query: " if i == q else f"{sims[i]:.2f} ") + parts_test.classes[labels[i]], fontsize=8); a.axis("off")

## 6. (Colab, optional) Zero-shot classification with CLIP

**CLIP** was trained on 400 M image–text pairs. It can classify images into classes described in **plain text**, without any training example. Run this cell on Colab (it downloads ~600 MB).

In [ ]:
RUN_CLIP = False
if RUN_CLIP:
    from transformers import CLIPModel, CLIPProcessor  # pip install transformers
    clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    texts = ["a photo of an ant", "a photo of a bee"]
    files = sorted((DATA / "hymenoptera_data" / "val").glob("*/*.jpg"))
    correct = 0
    for f in files:
        inputs = proc(text=texts, images=Image.open(f).convert("RGB"), return_tensors="pt", padding=True)
        pred = clip(**inputs).logits_per_image.argmax().item()
        correct += pred == (f.parent.name == "bees")
    print("CLIP zero-shot accuracy:", correct / len(files))
    # Try it on the flanges: "a metal part with a scratch", ... -- does it work? Why (not)?